In [1]:
import xarray as xr
import glob
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import dask
import numpy as np

In [31]:
cluster = PBSCluster(
    cores=1,
    memory='32GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='2:00:00'
)
cluster.scale(jobs=15)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.97:39949,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


2026-07-06 11:50:17,276 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyterhub.hpc.ucar.edu', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the 

In [2]:
years = np.arange(1981, 2027, 1)
years

array([1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991,
       1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002,
       2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
       2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024,
       2025, 2026])

In [28]:
path = '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/'
test_ds = xr.open_dataset(path+'194001/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940011606_1940020106.nc')
test_ds

<xarray.Dataset> Size: 2GB
Dimensions:                (forecast_initial_time: 32, forecast_hour: 12,
                            latitude: 721, longitude: 1440)
Coordinates:
  * forecast_initial_time  (forecast_initial_time) datetime64[ns] 256B 1940-0...
  * forecast_hour          (forecast_hour) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude               (latitude) float64 6kB 90.0 89.75 ... -89.75 -90.0
  * longitude              (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
Data variables:
    MVIMD                  (forecast_initial_time, forecast_hour, latitude, longitude) float32 2GB ...
    utc_date               (forecast_initial_time) int32 128B ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.8.1
    CONVERSION_PLATFORM:  Linux r3i0n4 4.12.14-95.51-default #1 SMP Fri Apr 1...
    CONVERSION_DATE:      Thu Mar 16 18:26:34 MDT 2023
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Thu Mar 16 18:26:43 2023: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 5.0.3 (Homepage = http://n...

In [4]:
test2_ds = xr.open_dataset('/gdex/data/d633000/e5.oper.an')
test2_ds

<xarray.Dataset> Size: 1GB
Dimensions:                (forecast_initial_time: 30, forecast_hour: 12,
                            latitude: 721, longitude: 1440)
Coordinates:
  * forecast_initial_time  (forecast_initial_time) datetime64[ns] 240B 1940-0...
  * forecast_hour          (forecast_hour) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude               (latitude) float64 6kB 90.0 89.75 ... -89.75 -90.0
  * longitude              (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
Data variables:
    VIMD                   (forecast_initial_time, forecast_hour, latitude, longitude) float32 1GB ...
    utc_date               (forecast_initial_time) int32 120B ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.8.1
    CONVERSION_PLATFORM:  Linux r10i5n20 4.12.14-95.51-default #1 SMP Fri Apr...
    CONVERSION_DATE:      Thu Mar 16 17:38:29 MDT 2023
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Thu Mar 16 17:38:37 2023: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 5.0.3 (Homepage = http://n...

In [30]:
mvimd_files = sorted(glob.glob(path+'*/*054_mvimd*.nc'))
mvimd_files

['/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194001/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940010106_1940011606.nc',
 '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194001/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940011606_1940020106.nc',
 '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194002/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940020106_1940021606.nc',
 '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194002/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940021606_1940030106.nc',
 '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194003/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940030106_1940031606.nc',
 '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194003/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940031606_1940040106.nc',
 '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194004/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940040106_1940041606.nc',
 '/gdex/data/d633000/e5.oper.fc.sfc.meanflux/194004/e5.oper.fc.sfc.meanflux.235_054_mvimd.ll025sc.1940041606_1940050106.nc',


In [36]:
def crop_mvimd_ds(ds):
    ds = ds['MVIMD']
    ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180
    ds = ds.sortby(ds.longitude)
    ds = ds.sortby(ds.latitude)
    ds_crop = ds.sel(latitude=slice(7.75, 28), longitude=slice(-89, -59))
    ds_crop = ds_crop.sel(forecast_initial_time=ds_crop.forecast_initial_time.dt.year.isin(years))
    return ds_crop


MVIMD = xr.open_mfdataset(mvimd_files, combine='nested', concat_dim='forecast_initial_time',
                     engine='h5netcdf', parallel=True,
                     preprocess=crop_mvimd_ds,
                     chunks={})
MVIMD

<xarray.DataArray 'MVIMD' (forecast_initial_time: 33052, forecast_hour: 12,
                           latitude: 82, longitude: 121)> Size: 16GB
dask.array<concatenate, shape=(33052, 12, 82, 121), dtype=float32, chunksize=(1, 12, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * forecast_initial_time  (forecast_initial_time) datetime64[ns] 264kB 1981-...
  * forecast_hour          (forecast_hour) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
Attributes: (12/14)
    long_name:                                          Mean vertically integ...
    short_name:                                         mvimd
    units:                                              kg m**-2 s**-1
    original_format:                                    WMO GRIB 1 with ECMWF...
    ecmwf_local_table:                                  235
    ecmwf_parameter:                                    54
    ...                                                 ...
    grid_specification:                                 0.25 degree x 0.25 de...
    rda_dataset:                                        ds633.0
    rda_dataset_url:                                    https:/rda.ucar.edu/d...
    rda_dataset_doi:                                    DOI: 10.5065/BH6N-5N20
    rda_dataset_group:                                  ERA5 atmospheric surf...
    QuantizeGranularBitGroomNumberOfSignificantDigits:  7

In [38]:
MVIMD = MVIMD.chunk({'latitude': -1, 'longitude': -1, 'forecast_initial_time': 8263})
MVIMD

<xarray.DataArray 'MVIMD' (forecast_initial_time: 33052, forecast_hour: 12,
                           latitude: 82, longitude: 121)> Size: 16GB
dask.array<rechunk-merge, shape=(33052, 12, 82, 121), dtype=float32, chunksize=(8263, 12, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * forecast_initial_time  (forecast_initial_time) datetime64[ns] 264kB 1981-...
  * forecast_hour          (forecast_hour) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude               (latitude) float64 656B 7.75 8.0 8.25 ... 27.75 28.0
  * longitude              (longitude) float64 968B -89.0 -88.75 ... -59.0
Attributes: (12/14)
    long_name:                                          Mean vertically integ...
    short_name:                                         mvimd
    units:                                              kg m**-2 s**-1
    original_format:                                    WMO GRIB 1 with ECMWF...
    ecmwf_local_table:                                  235
    ecmwf_parameter:                                    54
    ...                                                 ...
    grid_specification:                                 0.25 degree x 0.25 de...
    rda_dataset:                                        ds633.0
    rda_dataset_url:                                    https:/rda.ucar.edu/d...
    rda_dataset_doi:                                    DOI: 10.5065/BH6N-5N20
    rda_dataset_group:                                  ERA5 atmospheric surf...
    QuantizeGranularBitGroomNumberOfSignificantDigits:  7

In [40]:
MVIMD.to_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/MVIMD', mode='w')

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3415: UserWarning: Sending large graph of size 21.52 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
